# Adaptive Agent Analysis

Comprehensive analysis of the Neural LSTM-based Adaptive Agent performance:
- Inference time and computational efficiency
- Win rates against all opponents (with confidence intervals)
- Adversary identification speed and accuracy
- Performance breakdown by player position and opponent type

## 1. Import Required Libraries

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Tuple
from scipy import stats
import time

# Setup paths
sys.path.insert(0, str(Path().cwd().parent))

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully")

Libraries imported successfully


## 2. Load Agent and Setup Game Environment

In [2]:
import importlib
import sys

# Reload modules to pick up code changes
modules_to_reload = [
    'liars_dice.agents.adapter_agent.adaptive_training',
    'liars_dice.agents.adapter_agent',
    'liars_dice.agents.adapter_agent.action_tracker',
    'liars_dice.agents.adaptive_agent',
]
for module_name in modules_to_reload:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from liars_dice.core.config import GameConfig
from liars_dice.core.engine import GameEngine
from liars_dice.agents.adaptive_agent import AdaptiveAgent
from liars_dice.agents.ppo_agent import PPOAgent
from liars_dice.agents import AGENT_MAP
from liars_dice.agents.adaptive_agent_utils.config import PATH_CONFIG
from liars_dice.agents.adaptive_agent_utils.action_tracker import ActionTrackerWrapper

print("="*60)
print("AVAILABLE AGENTS IN AGENT_MAP")
print("="*60)
print(f"Total agents in AGENT_MAP: {len(AGENT_MAP)}")
print(f"Agent keys: {sorted(AGENT_MAP.keys())}\n")

# Build opponent classes (match train_adaptive_agent.py logic)
opponent_classes = {}
for agent_name, agent_cls in AGENT_MAP.items():
    if agent_name not in ["rl_ppo", "adaptive"]:  # Exclude base RL and adaptive
        try:
            _ = agent_cls()
            opponent_classes[agent_cls.__name__] = agent_cls
        except Exception:
            pass

# Add the frozen generalist PPO as an opponent
class GeneralistPPOAgent(PPOAgent):
    """Frozen generalist PPO model as a trainable opponent."""
    def __init__(self):
        super().__init__(model_path=PATH_CONFIG["generalist_model"], stochastic=False)

opponent_classes["GeneralistPPO"] = GeneralistPPOAgent

print("="*60)
print("OPPONENT CLASSES FOR ANALYSIS")
print("="*60)
print(f"Total opponent classes: {len(opponent_classes)}")
print(f"Opponents: {sorted(opponent_classes.keys())}\n")

# Verify all trainable agents are included (compare by class name)
available_class_names = {cls.__name__ for key, cls in AGENT_MAP.items() if key not in ["rl_ppo", "adaptive"]}
missing = available_class_names - set(opponent_classes.keys())
if missing:
    print(f"⚠️  Warning: These agent classes are not in opponent_classes: {sorted(missing)}")
else:
    print("✓ All available agents are included in opponent_classes")

# Initialize adaptive agent
try:
    adaptive_agent = AdaptiveAgent(device="cpu")
    print(f"\n✓ Adaptive Agent loaded successfully")
except Exception as e:
    print(f"Error loading Adaptive Agent: {e}")

AVAILABLE AGENTS IN AGENT_MAP
Total agents in AGENT_MAP: 30
Agent keys: ['adaptive', 'aggressive', 'alternator', 'bayesian', 'bluffing', 'chaotic', 'chaotic_safe', 'chaotic_unsafe', 'conservative', 'cycleface', 'hat_adapter_agent', 'maxcount', 'maxraise', 'minraise', 'mirror', 'nash_cfr', 'onesarewild', 'parity', 'probability_maxraise', 'probability_minraise', 'random', 'random_aggressive', 'random_cautious', 'random_facefixed', 'random_facerandom', 'randomface', 'randomthreshold', 'rl_ppo', 'safeface', 'thresholdliar']

OPPONENT CLASSES FOR ANALYSIS
Total opponent classes: 29
Opponents: ['AggressiveAgent', 'AggressiveRandomAgent', 'AlternatorAgent', 'BayesianAgent', 'BluffingAgent', 'CautiousRandomAgent', 'ChaoticAgent', 'ChaoticSafeAgent', 'ChaoticUnsafeAgent', 'ConservativeAgent', 'CycleFaceAgent', 'FaceFixedRandomAgent', 'FaceRandomRandomAgent', 'GeneralistPPO', 'HatAdapterAgent', 'MaxCountBidAgent', 'MaxRaiseAgent', 'MinRaiseAgent', 'MirrorAgent', 'NashCFRAgent', 'OnesAreWildAgent

## 3. Game Metrics Collection Functions

In [3]:
class AdaptiveAgentMetrics:
    """Collects metrics during adaptive agent gameplay."""
    
    def __init__(self):
        self.inference_times = []  # Time per action selection
        self.steps_to_identification = None  # When was opponent identified
        self.predicted_opponent = None  # Predicted opponent (highest confidence)
        self.actual_opponent = None  # Ground truth opponent
        self.identification_correct = None  # Was prediction correct
        self.belief_trajectory = []  # Belief distribution over time
        self.won = None  # Did adaptive agent win
        self.was_player_0 = None  # Was adaptive agent player 0

def play_game_with_metrics(
    adaptive_agent: AdaptiveAgent,
    opponent_cls,
    opponent_name: str,
    game_config: GameConfig,
    adaptive_is_player_0: bool = True
) -> Tuple[AdaptiveAgentMetrics, bool]:
    """
    Play a game and collect metrics from the adaptive agent.
    IMPORTANT: Uses ActionTrackerWrapper to properly capture opponent actions for belief updating.
    
    Returns:
        metrics: AdaptiveAgentMetrics object
        adaptive_won: Whether adaptive agent won
    """    
    metrics = AdaptiveAgentMetrics()
    metrics.actual_opponent = opponent_name
    metrics.was_player_0 = adaptive_is_player_0
    
    # Setup agents
    opponent = opponent_cls()
    adaptive_agent.reset()
    
    # Wrap opponent to track its actions
    wrapped_opponent = ActionTrackerWrapper(opponent, adaptive_agent=adaptive_agent)
    
    if adaptive_is_player_0:
        agents = [adaptive_agent, wrapped_opponent]
        adaptive_player = 0
    else:
        agents = [wrapped_opponent, adaptive_agent]
        adaptive_player = 1
    
    # Play match with dice elimination
    engine = GameEngine(game_config)
    dice_counts = [game_config.total_dice, game_config.total_dice]
    
    steps = 0
    identified_at_step = None
    
    while min(dice_counts) > 0:
        # Update dice counts
        for i in range(2):
            engine.state.players[i].num_dice = dice_counts[i]
        engine.start_new_round()
        
        # Play round
        while not engine.is_terminal():
            current_player = engine.state.public.current_player
            view = engine.get_view(current_player)
            
            # Measure inference time for adaptive agent
            if current_player == adaptive_player:
                start_time = time.time()
                action = agents[current_player].choose_action(view)
                inference_time = time.time() - start_time
                metrics.inference_times.append(inference_time)
                steps += 1
            else:
                # Opponent turn - action will be tracked via ActionTrackerWrapper
                action = agents[current_player].choose_action(view)
            
            try:
                engine.apply_action(current_player, action)
            except Exception:
                break
            
            # Check identification status AFTER each action (both players)
            # This ensures we capture when beliefs update from opponent actions
            if identified_at_step is None:
                belief_summary = adaptive_agent.get_belief_summary()
                if not belief_summary['using_generalist']:
                    identified_at_step = steps
                metrics.belief_trajectory.append(belief_summary['beliefs'].copy())
        
        # Round ended
        if engine.is_terminal():
            round_loser = engine.state.public.loser
            dice_counts[round_loser] -= 1
    
    # Determine winner
    winner = 0 if dice_counts[0] > 0 else 1
    metrics.won = (winner == adaptive_player)
    metrics.steps_to_identification = identified_at_step if identified_at_step else steps
    
    # Final prediction: Get the opponent with highest belief (predicted opponent)
    final_beliefs = adaptive_agent.get_belief_summary()
    beliefs_dict = final_beliefs['beliefs']
    
    # Find opponent with maximum belief
    if beliefs_dict:
        metrics.predicted_opponent = max(beliefs_dict.items(), key=lambda x: x[1])[0]
    else:
        metrics.predicted_opponent = None
    
    metrics.identification_correct = (metrics.predicted_opponent == opponent_name)
    
    return metrics

print("Metrics collection functions defined")

Metrics collection functions defined


## 4. Run Games and Collect Data

In [4]:
# Configuration
GAMES_PER_OPPONENT = 200
game_config = GameConfig(
    num_players=2,
    total_dice=5,
    faces=(1, 2, 3, 4, 5, 6),
    ones_wild=False
)

# Create single adaptive agent instance (reuse, reset between games)
adaptive_agent_instance = AdaptiveAgent(device="cpu")

# Collect results
all_results = []

print(f"Running {GAMES_PER_OPPONENT} games per opponent ({len(opponent_classes)} opponents)...\n")

for opp_idx, (opp_name, opp_cls) in enumerate(sorted(opponent_classes.items())):
    print(f"[{opp_idx+1}/{len(opponent_classes)}] Testing against {opp_name}...")
    
    for game_num in range(GAMES_PER_OPPONENT):
        # Alternate who goes first
        adaptive_is_player_0 = (game_num % 2 == 0)
        
        try:
            # Reset agent state between games (clears LSTM + beliefs)
            adaptive_agent_instance.reset()
            
            metrics = play_game_with_metrics(
                adaptive_agent_instance,
                opp_cls,
                opp_name,
                game_config,
                adaptive_is_player_0
            )
            
            # Store results
            result = {
                'opponent': opp_name,
                'game_num': game_num,
                'adaptive_is_player_0': adaptive_is_player_0,
                'adaptive_won': metrics.won,
                'avg_inference_time': np.mean(metrics.inference_times),
                'max_inference_time': np.max(metrics.inference_times),
                'min_inference_time': np.min(metrics.inference_times),
                'steps_to_identification': metrics.steps_to_identification,
                'identification_correct': metrics.identification_correct,
                'predicted_opponent': metrics.predicted_opponent,
            }
            all_results.append(result)
        except Exception as e:
            print(f"  Error in game {game_num}: {e}")
    
    print(f"  ✓ Completed {GAMES_PER_OPPONENT} games\n")

# Convert to DataFrame
df_results = pd.DataFrame(all_results)
print(f"\n✓ Collected {len(df_results)} game results")
print(f"\nDataFrame shape: {df_results.shape}")
print(df_results.head())

Running 200 games per opponent (29 opponents)...

[1/29] Testing against AggressiveAgent...
  ✓ Completed 200 games

[2/29] Testing against AggressiveRandomAgent...
  ✓ Completed 200 games

[3/29] Testing against AlternatorAgent...
  ✓ Completed 200 games

[4/29] Testing against BayesianAgent...
  ✓ Completed 200 games

[5/29] Testing against BluffingAgent...
  ✓ Completed 200 games

[6/29] Testing against CautiousRandomAgent...
  ✓ Completed 200 games

[7/29] Testing against ChaoticAgent...
  ✓ Completed 200 games

[8/29] Testing against ChaoticSafeAgent...
  ✓ Completed 200 games

[9/29] Testing against ChaoticUnsafeAgent...
  ✓ Completed 200 games

[10/29] Testing against ConservativeAgent...
  ✓ Completed 200 games

[11/29] Testing against CycleFaceAgent...
  ✓ Completed 200 games

[12/29] Testing against FaceFixedRandomAgent...
  ✓ Completed 200 games

[13/29] Testing against FaceRandomRandomAgent...
  ✓ Completed 200 games

[14/29] Testing against GeneralistPPO...
  ✓ Completed 2

## 5. Calculate Average Inference Time with Confidence Intervals

In [14]:
def calculate_ci(data, confidence=0.95):
    """Calculate confidence interval for a dataset."""
    n = len(data)
    mean = np.mean(data)
    std_err = stats.sem(data)
    ci = std_err * stats.t.ppf((1 + confidence) / 2, n - 1)
    return mean, ci

# Overall inference time statistics
all_inference_times = df_results['avg_inference_time'].values
mean_time = np.mean(all_inference_times)
std_time = np.std(all_inference_times)

print("="*60)
print("INFERENCE TIME ANALYSIS")
print("="*60)
print(f"Mean ± Std: {mean_time*1000:.4f} ± {std_time*1000:.4f} ms")
print(f"Min: {np.min(all_inference_times)*1000:.4f} ms")
print(f"Max: {np.max(all_inference_times)*1000:.4f} ms")

# Per-opponent inference time
print("\n" + "="*60)
print("INFERENCE TIME BY OPPONENT")
print("="*60)

inference_by_opponent = []
for opp in sorted(df_results['opponent'].unique()):
    opp_times = df_results[df_results['opponent'] == opp]['avg_inference_time'].values
    mean, ci = calculate_ci(opp_times)
    mean = np.mean(opp_times)
    std = np.std(opp_times)
    inference_by_opponent.append({
        'Opponent': opp,
        'Mean (ms)': mean * 1000,
        'Std (ms)': std * 1000,
        'N': len(opp_times)
    })

df_inference = pd.DataFrame(inference_by_opponent)
print(df_inference.to_string(index=False))

INFERENCE TIME ANALYSIS
Mean ± Std: 0.7041 ± 0.8434 ms
Min: 0.0000 ms
Max: 7.1245 ms

INFERENCE TIME BY OPPONENT
                Opponent  Mean (ms)  Std (ms)   N
         AggressiveAgent   0.626392  0.683012 200
   AggressiveRandomAgent   0.601693  0.813655 200
         AlternatorAgent   0.793885  0.943810 200
           BayesianAgent   0.562185  0.689095 200
           BluffingAgent   0.669981  0.977153 200
     CautiousRandomAgent   0.621932  0.933042 200
            ChaoticAgent   0.614125  0.889830 200
        ChaoticSafeAgent   0.520841  0.745820 200
      ChaoticUnsafeAgent   0.629385  0.783948 200
       ConservativeAgent   0.676892  0.680049 200
          CycleFaceAgent   0.649960  0.756928 200
    FaceFixedRandomAgent   0.636207  0.852865 200
   FaceRandomRandomAgent   0.641326  0.758642 200
           GeneralistPPO   0.603391  0.772293 200
         HatAdapterAgent   0.776161  0.885753 200
        MaxCountBidAgent   0.933444  0.943151 200
           MaxRaiseAgent   0.726472  

## 6. Analyze Win Rate Against Adversaries

In [15]:
# Overall win rate
total_wins = df_results['adaptive_won'].sum()
total_games = len(df_results)
overall_wr = total_wins / total_games

print("="*60)
print("OVERALL WIN RATE")
print("="*60)
print(f"Win Rate: {overall_wr:.1%}")
print(f"Wins/Games: {total_wins}/{total_games}")

# Win rate by player position
p0_games = df_results[df_results['adaptive_is_player_0'] == True]
p1_games = df_results[df_results['adaptive_is_player_0'] == False]

p0_wr = p0_games['adaptive_won'].sum() / len(p0_games)
p1_wr = p1_games['adaptive_won'].sum() / len(p1_games)

print(f"\nWin Rate as Player 0: {p0_wr:.1%} ({p0_games['adaptive_won'].sum()}/{len(p0_games)})")
print(f"Win Rate as Player 1: {p1_wr:.1%} ({p1_games['adaptive_won'].sum()}/{len(p1_games)})")

# Win rate by opponent with standard deviation
print("\n" + "="*60)
print("WIN RATE BY OPPONENT (Mean ± Std)")
print("="*60)

win_rate_data = []
for opp in sorted(df_results['opponent'].unique()):
    opp_games = df_results[df_results['opponent'] == opp]
    wins = opp_games['adaptive_won'].sum()
    total = len(opp_games)
    
    # Win rate mean and std
    p = wins / total
    # Std for binomial: sqrt(p*(1-p)/n)
    std = np.sqrt(p * (1 - p) / total)
    
    # Player position split
    opp_p0 = opp_games[opp_games['adaptive_is_player_0'] == True]
    opp_p1 = opp_games[opp_games['adaptive_is_player_0'] == False]
    p0_wr_opp = opp_p0['adaptive_won'].sum() / len(opp_p0) if len(opp_p0) > 0 else 0
    p1_wr_opp = opp_p1['adaptive_won'].sum() / len(opp_p1) if len(opp_p1) > 0 else 0
    
    win_rate_data.append({
        'Opponent': opp,
        'Win Rate': p,
        'Std': std,
        'P0 WR': p0_wr_opp,
        'P1 WR': p1_wr_opp,
    })

df_win_rates = pd.DataFrame(win_rate_data)
df_win_rates = df_win_rates.sort_values('Win Rate', ascending=False)

print(df_win_rates[['Opponent', 'Win Rate', 'Std', 'P0 WR', 'P1 WR']].to_string(index=False))

OVERALL WIN RATE
Win Rate: 97.8%
Wins/Games: 5672/5800

Win Rate as Player 0: 97.1% (2817/2900)
Win Rate as Player 1: 98.4% (2855/2900)

WIN RATE BY OPPONENT (Mean ± Std)
                Opponent  Win Rate      Std  P0 WR  P1 WR
         AggressiveAgent     1.000 0.000000   1.00   1.00
           BluffingAgent     1.000 0.000000   1.00   1.00
           BayesianAgent     1.000 0.000000   1.00   1.00
    FaceFixedRandomAgent     1.000 0.000000   1.00   1.00
     CautiousRandomAgent     1.000 0.000000   1.00   1.00
   FaceRandomRandomAgent     1.000 0.000000   1.00   1.00
       ConservativeAgent     1.000 0.000000   1.00   1.00
          CycleFaceAgent     1.000 0.000000   1.00   1.00
           MinRaiseAgent     1.000 0.000000   1.00   1.00
           MaxRaiseAgent     1.000 0.000000   1.00   1.00
        MaxCountBidAgent     1.000 0.000000   1.00   1.00
         HatAdapterAgent     1.000 0.000000   1.00   1.00
           GeneralistPPO     1.000 0.000000   1.00   1.00
             Pari

## 7. Analyze Steps Until Adversary Identification

In [6]:
print("="*60)
print("ADVERSARY IDENTIFICATION ACCURACY & SPEED")
print("="*60)

# Overall identification accuracy
total_games = len(df_results)
correct_ids = df_results['identification_correct'].sum()
accuracy = correct_ids / total_games

print(f"\nOverall Identification Accuracy:")
print(f"  Correct IDs: {correct_ids}/{total_games} ({accuracy:.1%})")

# Filter for correctly identified games only
df_correct = df_results[df_results['identification_correct'] == True]

if len(df_correct) > 0:
    steps_data_correct = df_correct['steps_to_identification'].values
    mean_steps = np.mean(steps_data_correct)
    std_steps = np.std(steps_data_correct)
    
    print(f"\nSpeed Statistics (Correctly Identified Games Only):")
    print(f"  Mean ± Std: {mean_steps:.2f} ± {std_steps:.2f} steps")
    print(f"  Median steps to ID: {np.median(steps_data_correct):.2f}")
    print(f"  Min: {np.min(steps_data_correct):.0f}, Max: {np.max(steps_data_correct):.0f}")
else:
    print("\n⚠️  No correctly identified games found")

# By opponent
print(f"\n" + "="*60)
print("IDENTIFICATION ACCURACY & SPEED BY OPPONENT")
print("="*60)

id_data = []
for opp in sorted(df_results['opponent'].unique()):
    opp_games = df_results[df_results['opponent'] == opp]
    opp_correct = opp_games[opp_games['identification_correct'] == True]
    
    total_opp = len(opp_games)
    correct_opp = len(opp_correct)
    accuracy_opp = correct_opp / total_opp if total_opp > 0 else 0
    
    # Calculate steps stats for correctly identified games only
    if correct_opp > 0:
        opp_steps_correct = opp_correct['steps_to_identification'].values
        mean_steps_opp = np.mean(opp_steps_correct)
        std_steps_opp = np.std(opp_steps_correct)
        median_steps_opp = np.median(opp_steps_correct)
    else:
        mean_steps_opp = float('nan')
        std_steps_opp = float('nan')
        median_steps_opp = float('nan')
    
    id_data.append({
        'Opponent': opp,
        'Accuracy': accuracy_opp,
        'Mean Steps': mean_steps_opp,
        'Std': std_steps_opp,
    })

df_id_analysis = pd.DataFrame(id_data)
df_id_analysis = df_id_analysis.sort_values('Accuracy', ascending=False)

# Format output
print(df_id_analysis.to_string(index=False, formatters={
    'Accuracy': lambda x: f"{x:.1%}",
    'Mean Steps': lambda x: f"{x:.2f}" if not np.isnan(x) else "N/A",
    'Std': lambda x: f"±{x:.2f}" if not np.isnan(x) else "N/A",
}))

ADVERSARY IDENTIFICATION ACCURACY & SPEED

Overall Identification Accuracy:
  Correct IDs: 2343/5800 (40.4%)

Speed Statistics (Correctly Identified Games Only):
  Mean ± Std: 2.76 ± 0.92 steps
  Median steps to ID: 2.00
  Min: 2, Max: 4

IDENTIFICATION ACCURACY & SPEED BY OPPONENT
                Opponent Accuracy Mean Steps   Std
         AggressiveAgent   100.0%       3.10 ±0.94
           BayesianAgent   100.0%       2.00 ±0.00
           MaxRaiseAgent   100.0%       3.00 ±1.00
         RandomFaceAgent    93.0%       3.00 ±1.00
            NashCFRAgent    92.5%       2.00 ±0.00
         HatAdapterAgent    79.0%       3.04 ±0.81
             MirrorAgent    72.0%       3.39 ±0.92
           BluffingAgent    69.5%       2.98 ±1.00
        MaxCountBidAgent    50.0%       2.00 ±0.00
           GeneralistPPO    50.0%       3.00 ±0.00
           SafeFaceAgent    50.0%       4.00 ±0.00
       ConservativeAgent    50.0%       2.00 ±0.00
          CycleFaceAgent    50.0%       2.00 ±0.00
   